# Notebook 03 · L1：网内自动力学与容忍带 δ

**L1 = L0 + 网内自动力学 + 容忍判据 δ**，对应 **PNAS 2019 的 M(x_i) 与 δ**。

关键点：节点不再只有"活/死"，而有连续的功能量 `x_i ∈ [0,1]`；
当它相对健康值掉出容忍带（`|1 − x_i/x̃_i| > δ`）就判失效。

⚠️ 符号提醒：PNAS 原文的自动力学记 `W(x_i)`，本项目为免与耦合张量 `W(t)` 撞名，
统一改写为 `M(x_i)`；本代码里叫"自动力学（auto-dynamics）"。

## 环境准备（每个 notebook 都要先跑这两格）

In [ ]:
"""环境准备：定位项目根目录、加 src 到搜索路径、设好画图参数。"""
import sys
from pathlib import Path

# 从当前工作目录往上找，直到看见 src/phase6_interdependent（这样在 notebooks/ 里也能跑）
root = Path.cwd()
while not (root / "src" / "phase6_interdependent").exists() and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root / "src"))

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

RESULTS = root / "results"
RESULTS.mkdir(exist_ok=True)
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("项目根目录：", root)

In [ ]:
# 公共底座：一对 BA 双层网络 + 骨架类（后面所有 notebook 都从这一格开始）
from phase6_interdependent.core import InterdependentNetwork

N, m = 300, 3
G_A = nx.barabasi_albert_graph(N, m, seed=42)   # 层 A：比如电力网
G_B = nx.barabasi_albert_graph(N, m, seed=43)   # 层 B：比如通信网
nodes_A = sorted(G_A.nodes())
print(f"双层网络就绪：N={N}, 每层边数={G_A.number_of_edges()}")

## 第 1 步

In [ ]:
from phase6_interdependent.core import InterdependentNetwork
from phase6_interdependent.L1_intra_layer_dynamics import (
    AutoDynamicsConfig, run_dynamics, healthy_reference, L1Rule
)

cfg = AutoDynamicsConfig(mu=0.4, lam=0.3, omega=0.3, delta=0.2)
x_ref = healthy_reference(G_A, G_B, cfg)
print("健康参考值 x_ref 的前 5 个：", list(x_ref["A"].values())[:5])

net1 = InterdependentNetwork(G_A, G_B)
victim = sorted(G_A.nodes())[0]
net1.kill("A", victim)
run_dynamics(net1, cfg)
xs = np.array(list(net1.x["A"].values()))
print(f"杀掉一个节点后：x 最小值={xs.min():.3f}，均值={xs.mean():.3f}")
plt.figure(figsize=(7, 4))
plt.hist(xs, bins=30)
plt.xlabel("功能量 x")
plt.ylabel("节点数")
plt.title("L1：局部故障后全网功能量的分布")
plt.savefig(RESULTS / "notebook_03_x_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 第 2 步

In [ ]:
# 退化性检验：δ=1（容忍带最宽，等于"没有动力学失效"）必须与 L0 完全一致
from phase6_interdependent.L0_static_dependency import percolation_curve

cfg_one = AutoDynamicsConfig(mu=0.4, lam=0.3, omega=0.3, delta=1.0)
x_ref_one = healthy_reference(G_A, G_B, cfg_one)

fractions = np.linspace(0.0, 0.3, 7)
rng = np.random.default_rng(42)
nodes_A = sorted(G_A.nodes())
S_L0, S_L1 = [], []
for f in fractions:
    k = int(round(len(nodes_A) * f))
    a, b = [], []
    for _ in range(3):
        failed = ([] if k == 0 else rng.choice(nodes_A, size=k, replace=False))
        n0 = InterdependentNetwork(G_A, G_B, seed=42)
        n1 = InterdependentNetwork(G_A, G_B, seed=42)
        a.append(n0.run(failed)["S_A"])
        b.append(n1.run(failed, rule=L1Rule(cfg_one, x_ref_one))["S_A"])
    S_L0.append(np.mean(a)); S_L1.append(np.mean(b))

print("δ=1 的 L1 曲线：", [round(v, 4) for v in S_L1])
print("L0 曲线：      ", [round(v, 4) for v in S_L0])
assert np.allclose(S_L1, S_L0, atol=1e-9), "退化性检验失败！δ=1 必须退回 L0"
print("✅ 退化性检验通过：δ=1 时 L1 ≡ L0")

## 第 3 步

In [ ]:
# δ 越小（容忍带越窄）⇒ 越脆弱
plt.figure(figsize=(8, 5))
for delta in [0.05, 0.2, 0.5, 1.0]:
    c = AutoDynamicsConfig(mu=0.4, lam=0.3, omega=0.3, delta=delta)
    xr = healthy_reference(G_A, G_B, c)
    rng = np.random.default_rng(42)
    curve = []
    for f in fractions:
        k = int(round(len(nodes_A) * f))
        trials = []
        for _ in range(3):
            failed = ([] if k == 0 else rng.choice(nodes_A, size=k, replace=False))
            n = InterdependentNetwork(G_A, G_B, seed=42)
            trials.append(n.run(failed, rule=L1Rule(c, xr))["S_A"])
        curve.append(np.mean(trials))
    plt.plot(fractions, curve, "o-", label=f"δ={delta}" + ("（=L0）" if delta == 1.0 else ""))
plt.xlabel("移除比例 f")
plt.ylabel("存活序参量 S")
plt.title("L1：容忍带 δ 对级联后果的影响")
plt.legend(); plt.grid(alpha=0.3)
plt.savefig(RESULTS / "notebook_03_delta_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

**思考题**

1. μ、λ 与 ω 分别代表什么？（自反馈回弹 / 网内扩散程度 / 跨层影响权重）
2. 为什么用"相对掉幅" `|1 − x/x̃|` 而不是绝对阈值 `x < c`？
3. δ=1 时 L1 退化回 L0 —— 这种"能证明自己包含上一级"的设计有什么用？